In [2]:
!pip install xgboost

   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   --- ------------------------------------ 6.8/69.5 MB 39.6 MB/s eta 0:00:02
   ------- -------------------------------- 13.4/69.5 MB 35.4 MB/s eta 0:00:02
   ---------- ----------------------------- 17.8/69.5 MB 32.2 MB/s eta 0:00:02
   --------------- ------------------------ 26.2/69.5 MB 32.8 MB/s eta 0:00:02
   --------------------- ------------------ 36.7/69.5 MB 38.0 MB/s eta 0:00:01
   ------------------------- -------------- 45.1/69.5 MB 37.2 MB/s eta 0:00:01
   ----------------------------- ---------- 51.9/69.5 MB 36.2 MB/s eta 0:00:01
   --------------------------------- ------ 58.7/69.5 MB 35.9 MB/s eta 0:00:01
   ---------------------------------------  69.5/69.5 MB 37.6 MB/s eta 0:00:01
   ---------------------------------------- 69.5/69.5 MB 35.2 MB/s  0:00:02


In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, StandardScaler
from preprocessing import * # Import all from your file

# Load and Filter Data
df = pd.read_csv("../data/used_cars.csv")
df['price'] = df['price'].astype(str).str.replace(r'[$,]', '', regex=True).astype(float)

# CRITICAL FIX: Filter outliers to focus on the 'normal' market
df = df[(df['price'] >= 2000) & (df['price'] <= 150000)]

X = df.drop(columns=['price'])
y = np.log1p(df['price']) # Log-transform the target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipelines
mileage_pipeline = Pipeline([
    ('cleaner', NumericalStringCleaner(variables=['milage'])),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

flag_cols = ['accident', 'clean_title'] 

# Re-define your flag_pipeline using the updated list
flag_pipeline = Pipeline([
    ('standardizer', CategoricalStandardizer(variables=flag_cols)),
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

year_pipeline = Pipeline([
    ('cleaner', NumericalStringCleaner(variables=['model_year'])),
    ('age_calc', AgeCalculator(current_year=2026)),
    ('imputer', SimpleImputer(strategy='median'))
])

brand_model_pipeline = Pipeline([
    ('std', CategoricalStandardizer(variables=['brand', 'model'])),
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('grouper', RareCategoryGrouper(variables=['brand', 'model'], threshold=1)),
    ('encoder', TargetEncoder())
])

transmission_pipeline = Pipeline([
    ('standardizer', TransmissionStandardizer(variables=['transmission'])),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Update your ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num_mileage', mileage_pipeline, ['milage']),
        ('num_year', year_pipeline, ['model_year']),
        ('brand_model', brand_model_pipeline, ['brand', 'model']),
        # Re-add these one at a time to check their impact
        ('cat_transmission', transmission_pipeline, ['transmission']), 
        ('flags', flag_pipeline, ['accident', 'clean_title']) # Only include these two key flags
    ],
    remainder='drop'
)

# Model
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=7, n_jobs=-1))
])

full_pipeline.fit(X_train, y_train)

# Predict and convert back from Log
y_pred = np.expm1(full_pipeline.predict(X_test))
y_test_orig = np.expm1(y_test)

from sklearn.metrics import r2_score
print(f"Final R2 Score: {r2_score(y_test_orig, y_pred)}")

NameError: name 'TransmissionStandardizer' is not defined

In [4]:
full_pipeline.fit(X_train, np.log1p(y_train))

# When predicting:
y_pred_log = full_pipeline.predict(X_test)
y_pred = np.expm1(y_pred_log)

In [5]:
from sklearn.metrics import r2_score
r2 = r2_score(y_test, y_pred)
r2


0.07867064043583827

In [2]:
# 1. Manually transform the test set to get the output DataFrame
X_processed = full_pipeline.named_steps['preprocessor'].transform(X_test)

# 2. Extract column names directly from that DataFrame
feature_names = X_processed.columns.tolist()

# 3. Get importances from the regressor
importances = full_pipeline.named_steps['regressor'].feature_importances_

# 4. Create and print the importance DataFrame
importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
print(importance_df.sort_values(by='importance', ascending=False).head(10).to_string())

             feature  importance
6   tmp_col_name_3_0    0.229302
0   tmp_col_name_0_0    0.137408
2   tmp_col_name_1_0    0.082429
16  tmp_col_name_4_8    0.065394
15  tmp_col_name_4_7    0.062133
13  tmp_col_name_4_5    0.061745
7   tmp_col_name_3_1    0.061575
12  tmp_col_name_4_4    0.055824
4   tmp_col_name_2_1    0.055108
5   tmp_col_name_2_2    0.047832


              feature  importance
17  tmp_col_name_2_14    0.143176
22   tmp_col_name_4_0    0.138788
7    tmp_col_name_2_4    0.087040
0    tmp_col_name_0_0    0.068146
31   tmp_col_name_5_7    0.058102
2    tmp_col_name_1_0    0.046670
29   tmp_col_name_5_5    0.043888
23   tmp_col_name_4_1    0.034838
9    tmp_col_name_2_6    0.034521
5    tmp_col_name_2_2    0.033915


In [15]:
# Extract the names from the preprocessor directly
feature_names = full_pipeline.named_steps['preprocessor'].get_feature_names_out()

# Create a clear mapping DataFrame
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': full_pipeline.named_steps['regressor'].feature_importances_
})

# Display the top 20 with real names
print(importance_df.sort_values(by='importance', ascending=False).head(20).to_string())

AttributeError: Estimator cleaner does not provide get_feature_names_out. Did you mean to call pipeline[:-1].get_feature_names_out()?

In [6]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from preprocessing import NumericalStringCleaner, AgeCalculator

# 0. Load and Pre-process Target
df = pd.read_csv("../data/used_cars.csv")
df['price'] = df['price'].astype(str).str.replace(r'[$,]', '', regex=True).astype(float)

# 1. Select only the two core features
X = df[['milage', 'model_year']]
y = np.log1p(df['price'])  # Applying Log-transform immediately

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Define specialized pipeline for these two
simple_preprocessor = ColumnTransformer(transformers=[
    ('mileage_pipe', Pipeline([
        ('cleaner', NumericalStringCleaner(variables=['milage'])),
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), ['milage']),
    
    ('year_pipe', Pipeline([
        ('cleaner', NumericalStringCleaner(variables=['model_year'])),
        ('age_calc', AgeCalculator(current_year=2026)),
        ('imputer', SimpleImputer(strategy='median'))
    ]), ['model_year'])
])

# 3. Full Simple Pipeline
full_pipeline = Pipeline([
    ('preprocessor', simple_preprocessor),
    ('regressor', xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6))
])

# 4. Train and Predict
full_pipeline.fit(X_train, y_train)

# Predict and Reverse Log (expm1)
y_pred_log = full_pipeline.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_test_orig = np.expm1(y_test)

print(f"R2 Score (Mileage + Age only): {r2_score(y_test_orig, y_pred)}")

R2 Score (Mileage + Age only): 0.03914383590844617


In [8]:
# 1. Manually run the preprocessor on your training data
X_train_transformed = full_pipeline.named_steps['preprocessor'].transform(X_train)

# 2. Since set_config(transform_output="pandas") is on, this IS a DataFrame
# You can now print it directly
print("--- Transformed X_train ---")
print(X_train_transformed.head())

# 3. Check column names and data types to ensure they are what you expect
print("\n--- Column Names ---")
print(X_train_transformed.columns.tolist())

print("\n--- Column Stats (Check for weird values) ---")
print(X_train_transformed.describe().loc[['mean', 'min', 'max']])

--- Transformed X_train ---
      tmp_col_name_0_0  tmp_col_name_1_0
2473          0.154635               8.0
1338         -1.048390               3.0
1613         -0.184876              10.0
1610          0.235722               9.0
2600         -0.595937               8.0

--- Column Names ---
['tmp_col_name_0_0', 'tmp_col_name_1_0']

--- Column Stats (Check for weird values) ---
      tmp_col_name_0_0  tmp_col_name_1_0
mean     -7.920768e-17         10.442158
min      -1.236848e+00          2.000000
max       6.486516e+00         52.000000


In [19]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import r2_score

# Ensure your preprocessing.py is in the same folder
from preprocessing import NumericalStringCleaner, AgeCalculator

# 1. Load and Clean Data
df = pd.read_csv("../data/used_cars.csv")
df['price'] = df['price'].astype(str).str.replace(r'[$,]', '', regex=True).astype(float)

# 2. Filter Outliers (The "Smoking Gun" for your low R2)
# Removing extreme outliers > $150k so the model can learn the normal market
df = df[(df['price'] >= 2000) & (df['price'] <= 150000)]

# 3. Define Features and Target
X = df[['milage', 'model_year', 'brand', 'model']]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Define Sub-Pipelines
mileage_pipe = Pipeline([
    ('cleaner', NumericalStringCleaner(variables=['milage'])),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

year_pipe = Pipeline([
    ('cleaner', NumericalStringCleaner(variables=['model_year'])),
    ('age_calc', AgeCalculator(current_year=2026)),
    ('imputer', SimpleImputer(strategy='median'))
])

brand_model_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# 5. Master Preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('milage', mileage_pipe, ['milage']),
    ('year', year_pipe, ['model_year']),
    ('brand_model', brand_model_pipe, ['brand', 'model'])
])

# 6. Full Pipeline
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6))
])

# 7. Train and Evaluate
# Using log transform on y helps further stabilize the model
full_pipeline.fit(X_train, np.log1p(y_train))

y_pred_log = full_pipeline.predict(X_test)
y_pred = np.expm1(y_pred_log)

print(f"Filtered Dataset Size: {len(df)}")
print(f"Final R2 Score: {r2_score(y_test, y_pred)}")

Filtered Dataset Size: 3891
Final R2 Score: 0.6393977875880735


In [18]:


# Check for unrealistic values
print("Price Stats:")
print(df['price'].describe())

print("\nModel Year Stats:")
print(df['model_year'].describe())

print("\nMileage Stats:")
print(df['milage'].describe())

Price Stats:
count    4.009000e+03
mean     4.455319e+04
std      7.871064e+04
min      2.000000e+03
25%      1.720000e+04
50%      3.100000e+04
75%      4.999000e+04
max      2.954083e+06
Name: price, dtype: float64

Model Year Stats:
count    4009.000000
mean     2015.515590
std         6.104816
min      1974.000000
25%      2012.000000
50%      2017.000000
75%      2020.000000
max      2024.000000
Name: model_year, dtype: float64

Mileage Stats:
count            4009
unique           2818
top       110,000 mi.
freq               16
Name: milage, dtype: object
